In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# ===========================================================================
# Notebook  : 02a_silver_contact
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02a_silver_contact
# Purpose   : bronze.{customer_id}.crm_contacts → hgi.silver.contacts
# 
# Serverless : Works on 2X-Small (safe_spark_conf skips unsupported configs)
# Job Compute: Full perf configs applied automatically
# ===========================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:

%run ./silver_cdf_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",      "cust_001")
dbutils.widgets.text("starting_version", "0")
# starting_version='0' → replays full bronze history on first run
# After first run, checkpoint takes over (widget ignored)
# Set to a specific version number to skip already-processed history

customer_id      = dbutils.widgets.get("customer_id").strip().lower()
starting_version = dbutils.widgets.get("starting_version").strip()

In [0]:
# CELL 3 ── Object-specific constants
source_system = "salesforce"
object_name   = "contact"
load_type     = "incremental"

In [0]:
# CELL 4 ── Path and table resolution via pipeline_config helpers
bronze_full  = bronze_table(customer_id, object_name)
# bronze_full = bronze.{customer_id}.crm_contacts

target_full  = silver_table(object_name)
# target_full = hgi.silver.contacts

chk_path = checkpoint_path("silver", "salesforce", customer_id, "contact")

print(f"=== 02a Silver CDF: SALESFORCE Contact ===")
print(f"  Customer         : {customer_id}")
print(f"  Bronze source    : {bronze_full}")
print(f"  Silver target    : {target_full}")
print(f"  Checkpoint       : {chk_path}")
print(f"  Starting version : {starting_version}")

In [0]:

# CELL 5 ── Create silver schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")
create_silver_table(target_full, "contacts")
# uses SILVER_DDL_COLUMNS['contacts'] from pipeline_config

In [0]:
# CELL 6 ── Enable CDF on bronze source table
enable_cdf(bronze_full)

In [0]:
# CELL 7 ── Run CDF stream
try:
    run_cdf_stream(bronze_full, target_full, "salesforce", "contact",
                   chk_path, starting_version)
except Exception as e:
    print(f"❌ CDF stream failed: {e}")
    log_audit(
        job_name       = "02a_silver_contact",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 8 ── Conditional Post-load maintenance
# FIX: Only run OPTIMIZE for large batches (>1000 rows), skip VACUUM on incremental runs
OPTIMIZE_THRESHOLD = 1000  # Only OPTIMIZE if more than this many rows were merged

try:
    # Get the count of rows affected in the last operation from table history
    last_op = spark.sql(f"DESCRIBE HISTORY {target_full} LIMIT 1").collect()
    rows_merged_this_run = 0
    if last_op and last_op[0]["operationMetrics"]:
        metrics = last_op[0]["operationMetrics"]
        rows_merged_this_run = int(metrics.get("numTargetRowsUpdated", 0)) + int(metrics.get("numTargetRowsInserted", 0))

    if rows_merged_this_run > OPTIMIZE_THRESHOLD:
        print(f"  Running OPTIMIZE ({rows_merged_this_run:,} rows merged > {OPTIMIZE_THRESHOLD} threshold)...")
        spark.sql(f"OPTIMIZE {target_full}")
        print(f"  OPTIMIZE complete")
    else:
        print(f"  Skipping OPTIMIZE ({rows_merged_this_run:,} rows merged <= {OPTIMIZE_THRESHOLD} threshold)")

    # VACUUM only on historical loads, not incremental (saves ~30-60s per run)
    if load_type == "historical":
        print(f"  Running VACUUM (historical load)...")
        spark.sql(f"VACUUM {target_full} RETAIN 168 HOURS")
        print(f"  VACUUM complete")
    else:
        print(f"  Skipping VACUUM (incremental load — schedule weekly instead)")

    print(f"Silver CDF complete: {target_full}")

except Exception as e:
    print(f"⚠️ Post-load maintenance warning (non-fatal): {e}")
    # Don't fail the job for OPTIMIZE/VACUUM issues

In [0]:
# CELL 8 ── Post-load OPTIMIZE
try:
    spark.sql(f"OPTIMIZE {target_full}")
    print(f"Silver CDF complete: {target_full}")
except Exception as e:
    print(f"❌ OPTIMIZE failed: {e}")
    log_audit(
        job_name       = "02a_silver_contact",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"OPTIMIZE failed: {str(e)[:450]}",
    )
    raise